# CIFAR-100 CNN with VVTKDataset + VVTKDataLoader (no torch DataLoader)

This example uses our custom **VVTKDataLoader** (C++ based) instead of `torch.utils.data.DataLoader`.
The C++ loader handles batching, multi-threaded prefetch, and zero-copy reads directly from the VVTK shards.

1. Download CIFAR-100 from torchvision
2. Store train/test splits into VVTK sharded binary dataset (`./data`)
3. Train a simple CNN with Adam for 10 epochs using **VVTKDataLoader**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchvision import datasets
from vvtk_dataset import VVTKDataset, VVTKDataLoader

## Step 1 — Download CIFAR-100 and store into VVTK shards

In [ ]:
# Download CIFAR-100
train_cifar = datasets.CIFAR100(root='data/cifar100_raw', train=True,  download=True)
test_cifar  = datasets.CIFAR100(root='data/cifar100_raw', train=False, download=True)

print(f'Train samples: {len(train_cifar)}')
print(f'Test  samples: {len(test_cifar)}')
print(f'Image shape:   {np.array(train_cifar[0][0]).shape}')  # (32, 32, 3)

100%|██████████| 169M/169M [00:15<00:00, 11.0MB/s] 


Train samples: 50000
Test  samples: 10000
Image shape:   (32, 32, 3)


In [ ]:
def store_to_vvtk(cifar_dataset, path, num_shards=16):
    """Store a torchvision CIFAR dataset into VVTK binary shards.
    Each sample is a tuple of (image_uint8 [3,32,32], label_int64 [1]).
    """
    with VVTKDataset(path, mode='wb', num_shards=num_shards,
                     compression=['none', 'none']) as ds:
        for idx in range(len(cifar_dataset)):
            img_pil, label = cifar_dataset[idx]
            # HWC uint8 -> CHW uint8
            img = np.array(img_pil, dtype=np.uint8).transpose(2, 0, 1)
            lbl = np.array([label], dtype=np.int64)
            ds.add(idx, img, lbl)
    print(f'Saved {len(cifar_dataset)} samples to {path}')

store_to_vvtk(train_cifar, 'data/shards/train', num_shards=16)
store_to_vvtk(test_cifar,  'data/shards/test',  num_shards=4)

[VVTK] Saved dataset to ./data/train
Saved 50000 samples to ./data/train
[VVTK] Saved dataset to ./data/test
Saved 10000 samples to ./data/test


## Step 2 — Create VVTKDataLoader (replaces torch DataLoader)

`VVTKDataLoader` works directly on the `VVTKDataset` — no wrapper class needed.
It reads batches at the C++ level with multi-threaded prefetch.

Each iteration yields a list of `(data_tensor, length_tensor)` tuples, one per item in the sample:
- `batch[0]` → images `[B, 3, 32, 32]` as uint8, with their lengths
- `batch[1]` → labels `[B, 1]` as int64, with their lengths

In [ ]:
BATCH_SIZE = 128

# Open datasets in read mode
train_ds = VVTKDataset('data/shards/train', mode='rb', compression=['none', 'none'])
test_ds  = VVTKDataset('data/shards/test',  mode='rb', compression=['none', 'none'])

# VVTKDataLoader: specify the max shape, dtype and padding for each tensor in the tuple
# Item 0: image  -> shape (3, 32, 32), dtype uint8,  pad 0
# Item 1: label  -> shape (1,),        dtype int64,  pad 0
train_loader = VVTKDataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    num_workers=4,
    shapes=[(3, 32, 32), (1,)],
    dtypes=[torch.uint8, torch.int64]
)

test_loader = VVTKDataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    num_workers=4,
    shapes=[(3, 32, 32), (1,)],
    dtypes=[torch.uint8, torch.int64]
)

print(f'Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')

[VVTKLoader] Building index map...
[VVTKLoader] Ready in 0.02s
[VVTKLoader] Building index map...
[VVTKLoader] Ready in 0.00s
Train batches: 391  |  Test batches: 79


## Step 3 — Define a simple CNN

In [8]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 32x32 -> 16x16
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 16x16 -> 8x8
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 8x8 -> 4x4
            nn.Dropout(0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
print(f'Device: {device}  |  Params: {sum(p.numel() for p in model.parameters()):,}')

Device: cuda  |  Params: 838,148


## Step 4 — Train for 10 epochs

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 10

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch in train_loader:
        # VVTKDataLoader returns: [(img_tensor, img_len), (lbl_tensor, lbl_len)]
        imgs   = batch[0][0].float().div(255.0).to(device)   # [B, 3, 32, 32]
        labels = batch[1][0].squeeze(1).to(device)            # [B]

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += imgs.size(0)

    # --- Eval ---
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            imgs   = batch[0][0].float().div(255.0).to(device)
            labels = batch[1][0].squeeze(1).to(device)
            preds = model(imgs).argmax(1)
            test_correct += (preds == labels).sum().item()
            test_total += imgs.size(0)

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss/train_total:.4f}  Acc: {100*train_correct/train_total:.1f}% | '
          f'Test Acc: {100*test_correct/test_total:.1f}%')

Epoch  1/10 | Train Loss: 4.1844  Acc: 5.5% | Test Acc: 10.6%
Epoch  2/10 | Train Loss: 3.7187  Acc: 11.1% | Test Acc: 17.8%
Epoch  3/10 | Train Loss: 3.4699  Acc: 14.7% | Test Acc: 21.4%
Epoch  4/10 | Train Loss: 3.2884  Acc: 17.8% | Test Acc: 22.4%
Epoch  5/10 | Train Loss: 3.1508  Acc: 20.2% | Test Acc: 27.4%
Epoch  6/10 | Train Loss: 3.0366  Acc: 21.9% | Test Acc: 28.3%
Epoch  7/10 | Train Loss: 2.9535  Acc: 23.5% | Test Acc: 32.8%
Epoch  8/10 | Train Loss: 2.8772  Acc: 24.9% | Test Acc: 34.0%
Epoch  9/10 | Train Loss: 2.8093  Acc: 26.3% | Test Acc: 35.2%
Epoch 10/10 | Train Loss: 2.7443  Acc: 27.7% | Test Acc: 36.9%
